# Reprodukcja wyniku CLIP (zero-shot, CIFAR-10 / CIFAR-100)

**Analiza danych obrazowych i multimedialnych** · Politechnika Warszawska, Wydział Elektryczny

**Zespół 9:** Tymur Rozhkovskyi · Alina Upyrova · Radosław Tkaczyk

**Artykuł:** Radford A. i in. (2021). *Learning Transferable Visual Models From Natural Language Supervision*. ICML 2021. https://arxiv.org/abs/2103.00020
**Kod:** https://github.com/openai/CLIP (licencja MIT)

## Cel
Odtworzenie wyniku **zero-shot** modelu **CLIP ViT-B/32** na oficjalnych zbiorach testowych **CIFAR-10** i **CIFAR-100**
podanego w Tabeli 11 artykułu:

| Zbiór | Model | Top-1 accuracy (artykuł) |
|---|---|---|
| CIFAR-10 | CLIP ViT-B/32 | 91,3% |
| CIFAR-100 | CLIP ViT-B/32 | 65,1% |

## Struktura notatnika
0. Środowisko, wersje bibliotek i konfiguracja
1. Model i preprocessing
2. Dane (pobierane z kodu przez `torchvision`)
3. Prompty – 18 szablonów i nazwy klas z `data/prompts.md`
4. Klasyfikator zero-shot z embeddingów tekstu
5. Embeddingi obrazów *(komórka kosztowna – wynik zapisywany w `cache/`)*
6. Predykcja i metryki: top-1 accuracy, macro-F1, różnica względem artykułu, 95% CI (bootstrap)
7. Porównanie z publikacją
8. Analiza różnic i błędów
9. Podsumowanie

**Uruchomienie:** `Kernel → Restart & Run All`. Notatnik nie wymaga ręcznych kroków; dane i wagi pobierają się automatycznie
przy pierwszym uruchomieniu (ok. 160 MB danych + 338 MB wag). Wszystkie ścieżki są względne.

## 0. Środowisko, wersje bibliotek i konfiguracja

Instalacja zależności (jednorazowo, w terminalu, w katalogu projektu):
```bash
pip install -r requirements.txt
```

In [1]:
# Jeśli biblioteki nie są zainstalowane, odkomentuj poniższą linię (instalacja jednorazowa).
# %pip install -r requirements.txt

In [2]:
import json
import os 
import platform
import random
import time
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import torch
import torchvision 
from torchvision.datasets import CIFAR10, CIFAR100
from sklearn.metrics import f1_score, confusion_matrix
from tqdm.auto import tqdm

import clip  # oficjalna biblioteka openai/CLIP

# ---------------- Konfiguracja ----------------
CONFIG = {
    "model_name": "ViT-B/32",       # checkpoint autorów, bez dodatkowego trenowania
    "datasets": ["CIFAR10", "CIFAR100"],
    "split": "test",                # oficjalne zbiory testowe, po 10 000 obrazów
    "templates": "data/prompts.md (18 szablonów CIFAR)",
    "batch_size": 256,
    "num_workers": 0,               # 0 = najbezpieczniej w Jupyterze na macOS
    "precision": "float32",         # model ładowany na CPU (fp32), potem przenoszony na urządzenie
    "logit_scale": 100.0,           # jak w oficjalnym notebooku autorów; nie wpływa na argmax
    "bootstrap_samples": 1000,
    "seed": 42,
}
PAPER_TOP1 = {"CIFAR10": 91.3, "CIFAR100": 65.1}   # Tabela 11 artykułu, CLIP ViT-B/32 zero-shot

# Ścieżki względne
DATA_DIR = Path("data")
CACHE_DIR = Path("cache")
RESULTS_DIR = Path("results")
for d in (DATA_DIR, CACHE_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Ziarna losowe – inferencja jest deterministyczna, ziarno ma znaczenie tylko dla bootstrapu
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

# Urządzenie: CUDA -> Apple MPS (M1/M2/M3) -> CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

ENV = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "processor": platform.processor() or platform.machine(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "clip_models_available": clip.available_models(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "device": DEVICE,
}
print(json.dumps({"ENV": ENV, "CONFIG": CONFIG}, indent=2, ensure_ascii=False))
(RESULTS_DIR / "environment.json").write_text(
    json.dumps({"ENV": ENV, "CONFIG": CONFIG, "PAPER_TOP1": PAPER_TOP1}, indent=2, ensure_ascii=False)
)

{
  "ENV": {
    "python": "3.11.14",
    "platform": "macOS-15.3-arm64-arm-64bit",
    "processor": "arm",
    "torch": "2.15.0",
    "torchvision": "0.29.2",
    "clip_models_available": [
      "RN50",
      "RN101",
      "RN50x4",
      "RN50x16",
      "RN50x64",
      "ViT-B/32",
      "ViT-B/16",
      "ViT-L/14",
      "ViT-L/14@336px"
    ],
    "numpy": "2.4.6",
    "pandas": "3.1.1",
    "scikit-learn": "1.9.1",
    "matplotlib": "3.11.2",
    "device": "mps"
  },
  "CONFIG": {
    "model_name": "ViT-B/32",
    "datasets": [
      "CIFAR10",
      "CIFAR100"
    ],
    "split": "test",
    "templates": "data/prompts.md (18 szablonów CIFAR)",
    "batch_size": 256,
    "num_workers": 0,
    "precision": "float32",
    "logit_scale": 100.0,
    "bootstrap_samples": 1000,
    "seed": 42
  }
}


877

## 1. Model i preprocessing

Ładujemy oficjalny checkpoint **ViT-B/32** funkcją `clip.load(..., jit=False)`. Model ładujemy na CPU – biblioteka
konwertuje wtedy wagi do `float32` – a dopiero potem przenosimy go na wybrane urządzenie. Dzięki temu obliczenia są
w tej samej precyzji na CPU i na Apple MPS. (Na GPU CUDA autorzy używali `float16` – to jedna z możliwych drobnych przyczyn różnic.)

`preprocess` to dokładnie ta transformacja, której używali autorzy: `Resize(224, bicubic)` → `CenterCrop(224)` → `ToTensor` →
normalizacja średnią/odchyleniem CLIP. Nie zmieniamy jej.

In [3]:
t0 = time.time()
model, preprocess = clip.load(CONFIG["model_name"], device="cpu", jit=False)
model = model.float().to(DEVICE).eval()
print(f"Model załadowany w {time.time() - t0:.1f} s na urządzenie: {DEVICE}")

n_params = sum(p.numel() for p in model.parameters())
print(f"Liczba parametrów:        {n_params / 1e6:.1f} M")
print(f"Rozdzielczość wejścia:    {model.visual.input_resolution} px")
print(f"Długość kontekstu tekstu: {model.context_length} tokenów")
print(f"Wymiar wspólnej przestrzeni (embedding): {model.text_projection.shape[1]}")
print(f"Wyuczona temperatura: logit_scale = exp(t) = {model.logit_scale.exp().item():.2f}")
print("\nPreprocessing:\n", preprocess)

Model załadowany w 1.9 s na urządzenie: mps
Liczba parametrów:        151.3 M
Rozdzielczość wejścia:    224 px
Długość kontekstu tekstu: 77 tokenów
Wymiar wspólnej przestrzeni (embedding): 512
Wyuczona temperatura: logit_scale = exp(t) = 100.00

Preprocessing:
 Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    <function _convert_image_to_rgb at 0x14c4354e0>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)


## 2. Dane

Oficjalne zbiory testowe CIFAR-10 i CIFAR-100 (po 10 000 obrazów 32×32 px) pobierane przez `torchvision` do katalogu `data/`.
Sprawdzamy liczność zbiorów oraz to, że kolejność klas w `torchvision` jest taka sama jak w liście nazw klas autorów
(w CIFAR-100 `torchvision` używa podkreśleń, np. `aquarium_fish`, autorzy – spacji, np. `aquarium fish`).

In [4]:
DATASETS = {
    "CIFAR10": CIFAR10(root=DATA_DIR, train=False, download=True, transform=preprocess),
    "CIFAR100": CIFAR100(root=DATA_DIR, train=False, download=True, transform=preprocess),
}
for name, ds in DATASETS.items():
    assert len(ds) == 10_000, f"{name}: oczekiwano 10 000 obrazów, jest {len(ds)}"
    print(f"{name}: {len(ds)} obrazów testowych, {len(ds.classes)} klas")

 10%|███▋                                  | 16.4M/170M [03:25<32:16, 79.6kB/s]


KeyboardInterrupt: 

In [ ]:
# Wizualizacja: oryginalny obraz 32x32 i ten sam obraz po preprocessingu CLIP (224x224).
# Pokazuje, że model dostaje silnie powiększony obraz o niskiej rozdzielczości.
MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
STD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)

raw = CIFAR10(root=DATA_DIR, train=False, download=False)  # bez transformacji
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for i in range(6):
    img, y = raw[i]
    axes[0, i].imshow(img); axes[0, i].set_title(f"{raw.classes[y]}\n32×32", fontsize=9)
    x, _ = DATASETS["CIFAR10"][i]
    axes[1, i].imshow((x * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy())
    axes[1, i].set_title("po preprocess\n224×224", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "przyklady_preprocessing.png", dpi=120)
plt.show()

## 3. Prompty – szablony i nazwy klas autorów

Do reprodukcji używamy **dokładnie** nazw klas i 18 szablonów z pliku
[`data/prompts.md`](https://github.com/openai/CLIP/blob/main/data/prompts.md) z repozytorium autorów
(sekcje *CIFAR10* i *CIFAR100*, te same szablony dla obu zbiorów). Listy są skopiowane bez zmian, aby notatnik
nie zależał od dostępu do GitHuba. Prompty ustalamy **przed** liczeniem wyniku na zbiorze testowym.

In [ ]:
CIFAR_TEMPLATES = [
    "a photo of a {}.",
    "a blurry photo of a {}.",
    "a black and white photo of a {}.",
    "a low contrast photo of a {}.",
    "a high contrast photo of a {}.",
    "a bad photo of a {}.",
    "a good photo of a {}.",
    "a photo of a small {}.",
    "a photo of a big {}.",
    "a photo of the {}.",
    "a blurry photo of the {}.",
    "a black and white photo of the {}.",
    "a low contrast photo of the {}.",
    "a high contrast photo of the {}.",
    "a bad photo of the {}.",
    "a good photo of the {}.",
    "a photo of the small {}.",
    "a photo of the big {}.",
]

CLASSNAMES = {
    "CIFAR10": [
        "airplane", "automobile", "bird", "cat", "deer",
        "dog", "frog", "horse", "ship", "truck",
    ],
    "CIFAR100": [
        "apple", "aquarium fish", "baby", "bear", "beaver", "bed", "bee", "beetle", "bicycle", "bottle",
        "bowl", "boy", "bridge", "bus", "butterfly", "camel", "can", "castle", "caterpillar", "cattle",
        "chair", "chimpanzee", "clock", "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
        "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster", "house", "kangaroo", "keyboard",
        "lamp", "lawn mower", "leopard", "lion", "lizard", "lobster", "man", "maple tree", "motorcycle", "mountain",
        "mouse", "mushroom", "oak tree", "orange", "orchid", "otter", "palm tree", "pear", "pickup truck", "pine tree",
        "plain", "plate", "poppy", "porcupine", "possum", "rabbit", "raccoon", "ray", "road", "rocket",
        "rose", "sea", "seal", "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider",
        "squirrel", "streetcar", "sunflower", "sweet pepper", "table", "tank", "telephone", "television", "tiger", "tractor",
        "train", "trout", "tulip", "turtle", "wardrobe", "whale", "willow tree", "wolf", "woman", "worm",
    ],
}

assert len(CIFAR_TEMPLATES) == 18
for name, ds in DATASETS.items():
    tv_names = [c.replace("_", " ") for c in ds.classes]
    assert tv_names == CLASSNAMES[name], f"{name}: kolejność klas różni się od torchvision!"
    print(f"{name}: {len(CLASSNAMES[name])} nazw klas zgodnych z kolejnością etykiet torchvision ✓")

print("\nPrzykładowe prompty dla klasy 'cat':")
for t in CIFAR_TEMPLATES[:4]:
    print("  ", t.format("cat"))
print("   ...")

## 4. Klasyfikator zero-shot z embeddingów tekstu

Dla każdej klasy *c*:
1. wstawiamy nazwę klasy do każdego z 18 szablonów,
2. kodujemy teksty encoderem tekstu,
3. **normalizujemy L2** każdy embedding,
4. uśredniamy po szablonach (*prompt ensembling*),
5. ponownie normalizujemy L2.

Wynikiem jest macierz wag `W` o wymiarze `[512, liczba_klas]` – „klasyfikator liniowy” zbudowany wyłącznie z tekstu,
bez żadnego trenowania na danych docelowych. Procedura odpowiada oficjalnemu notebookowi autorów
(*Prompt_Engineering_for_ImageNet.ipynb*).

In [ ]:
@torch.no_grad()
def build_zeroshot_weights(classnames, templates):
    weights = []
    for classname in tqdm(classnames, desc="Embeddingi tekstu"):
        texts = [t.format(classname) for t in templates]
        tokens = clip.tokenize(texts).to(DEVICE)
        emb = model.encode_text(tokens).float()
        emb = emb / emb.norm(dim=-1, keepdim=True)    # L2 każdego promptu
        emb = emb.mean(dim=0)                         # ensemble po szablonach
        emb = emb / emb.norm()                        # ponowna normalizacja L2
        weights.append(emb)
    return torch.stack(weights, dim=1).cpu()          # [D, C]


ZS_WEIGHTS = {}
for name in CONFIG["datasets"]:
    ZS_WEIGHTS[name] = build_zeroshot_weights(CLASSNAMES[name], CIFAR_TEMPLATES)
    norms = ZS_WEIGHTS[name].norm(dim=0)
    assert torch.allclose(norms, torch.ones_like(norms), atol=1e-4), "Brak normalizacji L2!"
    print(f"{name}: macierz wag zero-shot {tuple(ZS_WEIGHTS[name].shape)}, normy kolumn = 1 ✓")

## 5. Embeddingi obrazów

**Komórka kosztowna obliczeniowo.** Kodowanie 2 × 10 000 obrazów encoderem ViT-B/32:
orientacyjnie **2–5 min** na Apple M3 (MPS), **10–20 min** na samym CPU, < 1 min na GPU CUDA.

Embeddingi (już znormalizowane L2) i etykiety zapisujemy do `cache/<zbiór>_ViT-B-32_test.pt`.
Kolejne uruchomienia notatnika wczytują je z dysku. Aby policzyć je od nowa, usuń pliki z `cache/`.

In [ ]:
@torch.no_grad()
def encode_images(dataset, name):
    cache_file = CACHE_DIR / f"{name}_{CONFIG['model_name'].replace('/', '-')}_test.pt"
    if cache_file.exists():
        blob = torch.load(cache_file)
        if blob.get("model_name") == CONFIG["model_name"] and blob["features"].shape[0] == len(dataset):
            print(f"{name}: wczytano embeddingi z {cache_file}")
            return blob["features"], blob["labels"]

    loader = torch.utils.data.DataLoader(
        dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"]
    )
    feats, labels = [], []
    t0 = time.time()
    for images, y in tqdm(loader, desc=f"Embeddingi obrazów {name}"):
        f = model.encode_image(images.to(DEVICE)).float()
        f = f / f.norm(dim=-1, keepdim=True)          # normalizacja L2
        feats.append(f.cpu())
        labels.append(y)
    feats, labels = torch.cat(feats), torch.cat(labels)
    elapsed = time.time() - t0
    torch.save({"features": feats, "labels": labels, "model_name": CONFIG["model_name"],
                "device": DEVICE, "seconds": elapsed}, cache_file)
    print(f"{name}: {feats.shape[0]} embeddingów w {elapsed:.1f} s → zapisano {cache_file}")
    return feats, labels


IMG_FEATS, LABELS = {}, {}
for name in CONFIG["datasets"]:
    IMG_FEATS[name], LABELS[name] = encode_images(DATASETS[name], name)
    n = IMG_FEATS[name].norm(dim=-1)
    assert torch.allclose(n, torch.ones_like(n), atol=1e-3), "Embeddingi obrazów nie są znormalizowane!"

## 6. Predykcja i metryki

* **logits** = `logit_scale · (obrazy · W)` – podobieństwo kosinusowe (wektory są znormalizowane), skalowane jak w CLIP;
  skala nie zmienia argmax, ma znaczenie tylko dla prawdopodobieństw (softmax).
* **Top-1 accuracy** – odsetek obrazów, dla których klasa o najwyższym podobieństwie jest poprawna (ta sama metryka co w Tabeli 11).
* **Macro-F1** – średnia F1 liczonych osobno dla każdej klasy (`sklearn.metrics.f1_score(average="macro")`).
* **Różnica** – nasze top-1 minus wynik z artykułu, w punktach procentowych (p.p.).
* **95% CI** – przedział ufności top-1 z bootstrapu (1000 losowań obrazów ze zwracaniem) – pozwala ocenić,
  czy różnica względem artykułu wykracza poza zmienność wynikającą ze skończonego zbioru testowego.
* Dodatkowo, pomocniczo: **top-5 accuracy**.

In [ ]:
def zero_shot_logits(feats, weights):
    return CONFIG["logit_scale"] * feats @ weights


def bootstrap_ci(correct, n_boot, seed, alpha=0.05):
    rng = np.random.default_rng(seed)
    n = len(correct)
    accs = np.array([correct[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    return np.percentile(accs, 100 * alpha / 2), np.percentile(accs, 100 * (1 - alpha / 2))


def evaluate(name, weights):
    logits = zero_shot_logits(IMG_FEATS[name], weights)
    y = LABELS[name].numpy()
    top5 = logits.topk(5, dim=1).indices.numpy()
    pred = top5[:, 0]
    correct = (pred == y).astype(float)
    lo, hi = bootstrap_ci(correct, CONFIG["bootstrap_samples"], CONFIG["seed"])
    return {
        "logits": logits, "pred": pred, "y": y,
        "top1": 100 * correct.mean(),
        "top5": 100 * (top5 == y[:, None]).any(axis=1).mean(),
        "macro_f1": 100 * f1_score(y, pred, average="macro"),
        "ci_low": 100 * lo, "ci_high": 100 * hi,
    }


RESULTS = {name: evaluate(name, ZS_WEIGHTS[name]) for name in CONFIG["datasets"]}

# Zapis surowych wyników: predykcja, top-5 i prawdopodobieństwo dla każdego obrazu
for name, r in RESULTS.items():
    probs = r["logits"].softmax(dim=-1)
    top5p, top5i = probs.topk(5, dim=1)
    classes = CLASSNAMES[name]
    pd.DataFrame({
        "image_idx": np.arange(len(r["y"])),
        "true_label": [classes[i] for i in r["y"]],
        "pred_label": [classes[i] for i in r["pred"]],
        "correct": r["pred"] == r["y"],
        "pred_prob": top5p[:, 0].numpy().round(4),
        "top5": ["|".join(classes[j] for j in row) for row in top5i.numpy()],
    }).to_csv(RESULTS_DIR / f"predictions_{name}.csv", index=False)
print("Zapisano surowe predykcje do results/predictions_*.csv")

## 7. Porównanie z publikacją

In [ ]:
summary = pd.DataFrame([
    {
        "Zbiór": name,
        "Model": "CLIP ViT-B/32",
        "Top-1 artykuł [%]": PAPER_TOP1[name],
        "Top-1 nasze [%]": round(r["top1"], 2),
        "Różnica [p.p.]": round(r["top1"] - PAPER_TOP1[name], 2),
        "95% CI top-1 [%]": f"{r['ci_low']:.2f} – {r['ci_high']:.2f}",
        "Artykuł w CI?": r["ci_low"] <= PAPER_TOP1[name] <= r["ci_high"],
        "Macro-F1 [%]": round(r["macro_f1"], 2),
        "Top-5 [%]": round(r["top5"], 2),
    }
    for name, r in RESULTS.items()
])
summary.to_csv(RESULTS_DIR / "summary_reprodukcja.csv", index=False)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(len(RESULTS))
names = list(RESULTS)
ours = [RESULTS[n]["top1"] for n in names]
err = np.array([[RESULTS[n]["top1"] - RESULTS[n]["ci_low"], RESULTS[n]["ci_high"] - RESULTS[n]["top1"]] for n in names]).T
ax.bar(x - 0.18, [PAPER_TOP1[n] for n in names], width=0.36, label="Artykuł (Tabela 11)", color="#9aa5b1")
ax.bar(x + 0.18, ours, width=0.36, yerr=err, capsize=4, label="Nasza reprodukcja (95% CI)", color="#1f6f8b")
for xi, v in zip(x, ours):
    ax.text(xi + 0.18, v + 1.5, f"{v:.1f}", ha="center", fontsize=9)
for xi, n in zip(x, names):
    ax.text(xi - 0.18, PAPER_TOP1[n] + 1.5, f"{PAPER_TOP1[n]:.1f}", ha="center", fontsize=9)
ax.set_xticks(x, names)
ax.set_ylabel("Top-1 accuracy [%]")
ax.set_ylim(0, 100)
ax.set_title("CLIP ViT-B/32 zero-shot: artykuł vs reprodukcja")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "porownanie_z_artykulem.png", dpi=120)
plt.show()

## 8. Analiza różnic i błędów

### 8.1 Kontrola wrażliwości: wpływ ensemble szablonów
Krótki test diagnostyczny (nie jest to jeszcze eksperyment z Konsultacji 3): porównujemy wynik z 18 szablonami
z wariantem z jednym szablonem `"a photo of a {}."`. Embeddingi obrazów są już w pamięci, więc liczy się to w kilka sekund.
Jeżeli ten jeden czynnik zmienia wynik o kilka p.p., to różnica rzędu ułamka p.p. względem artykułu
może wynikać z drobnych szczegółów implementacji.

In [ ]:
rows = []
for name in CONFIG["datasets"]:
    w_single = build_zeroshot_weights(CLASSNAMES[name], ["a photo of a {}."])
    r_single = evaluate(name, w_single)
    rows.append({"Zbiór": name,
                 "1 szablon: top-1 [%]": round(r_single["top1"], 2),
                 "18 szablonów: top-1 [%]": round(RESULTS[name]["top1"], 2),
                 "Zysk z ensemble [p.p.]": round(RESULTS[name]["top1"] - r_single["top1"], 2)})
sens = pd.DataFrame(rows)
sens.to_csv(RESULTS_DIR / "kontrola_ensemble.csv", index=False)
sens

### 8.2 CIFAR-10: macierz pomyłek

In [ ]:
r = RESULTS["CIFAR10"]
cm = confusion_matrix(r["y"], r["pred"], normalize="true") * 100
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=100)
cls = CLASSNAMES["CIFAR10"]
ax.set_xticks(range(10), cls, rotation=45, ha="right")
ax.set_yticks(range(10), cls)
for i in range(10):
    for j in range(10):
        if cm[i, j] >= 1:
            ax.text(j, i, f"{cm[i, j]:.0f}", ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > 50 else "black")
ax.set_xlabel("Predykcja"); ax.set_ylabel("Klasa prawdziwa")
ax.set_title("CIFAR-10: macierz pomyłek [% wiersza]")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cifar10_macierz_pomylek.png", dpi=120)
plt.show()

### 8.3 Dokładność per klasa i najczęstsze pomyłki

Sprawdzamy m.in. klasy o wieloznacznych nazwach w CIFAR-100 (`mouse`, `ray`, `tank`, `seal`, `plain`, `can`, `orange`),
które wskazaliśmy jako ryzyko w karcie projektu.

In [ ]:
def per_class_table(name):
    r = RESULTS[name]
    cls = CLASSNAMES[name]
    cm = confusion_matrix(r["y"], r["pred"], labels=range(len(cls)))
    acc = cm.diagonal() / cm.sum(axis=1) * 100
    off = cm.copy(); np.fill_diagonal(off, 0)
    top_conf = off.argmax(axis=1)
    return pd.DataFrame({
        "klasa": cls,
        "accuracy [%]": acc.round(1),
        "najczęściej mylona z": [cls[j] for j in top_conf],
        "liczba tych pomyłek": off.max(axis=1),
    }).sort_values("accuracy [%]")


pc10 = per_class_table("CIFAR10")
pc100 = per_class_table("CIFAR100")
pc10.to_csv(RESULTS_DIR / "per_class_CIFAR10.csv", index=False)
pc100.to_csv(RESULTS_DIR / "per_class_CIFAR100.csv", index=False)

print("CIFAR-10 – dokładność per klasa:")
display(pc10.reset_index(drop=True))
print("CIFAR-100 – 15 najsłabszych klas:")
display(pc100.head(15).reset_index(drop=True))

In [ ]:
AMBIGUOUS = ["mouse", "ray", "tank", "seal", "plain", "can", "orange", "bed", "table"]
print("CIFAR-100 – klasy o wieloznacznych nazwach:")
display(pc100[pc100["klasa"].isin(AMBIGUOUS)].reset_index(drop=True))
print(f"Średnia dokładność per klasa w całym CIFAR-100: {pc100['accuracy [%]'].mean():.1f}%")

In [ ]:
# Rozkład dokładności per klasa w CIFAR-100
fig, ax = plt.subplots(figsize=(12, 3.8))
s = pc100.sort_values("accuracy [%]")
ax.bar(range(len(s)), s["accuracy [%]"], color="#1f6f8b")
ax.axhline(RESULTS["CIFAR100"]["top1"], color="black", ls="--", lw=1, label="top-1 ogółem")
ax.set_xticks(range(len(s)), s["klasa"], rotation=90, fontsize=7)
ax.set_ylabel("accuracy [%]")
ax.set_title("CIFAR-100: dokładność zero-shot per klasa (posortowana)")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cifar100_per_class.png", dpi=120)
plt.show()

In [ ]:
# Przykłady błędnych predykcji (CIFAR-100)
raw100 = CIFAR100(root=DATA_DIR, train=False, download=False)
r = RESULTS["CIFAR100"]
wrong = np.where(r["pred"] != r["y"])[0]
rng = np.random.default_rng(CONFIG["seed"])
sample = rng.choice(wrong, size=12, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, i in zip(axes.ravel(), sample):
    img, y = raw100[i]
    ax.imshow(img)
    ax.set_title(f"prawda: {CLASSNAMES['CIFAR100'][y]}\npred: {CLASSNAMES['CIFAR100'][r['pred'][i]]}",
                 fontsize=8, color="darkred")
    ax.axis("off")
plt.suptitle("CIFAR-100: losowe błędne predykcje")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cifar100_bledy.png", dpi=120)
plt.show()

### 8.4 Możliwe przyczyny różnic względem artykułu

| Czynnik | U autorów | U nas | Oczekiwany wpływ |
|---|---|---|---|
| Precyzja obliczeń | `float16` na GPU (domyślne w `clip.load` dla CUDA) | `float32` na CPU / Apple MPS | bardzo mały (ułamek p.p.), inne zaokrąglenia mogą zmienić predykcję dla obrazów „na granicy” |
| Sprzęt i biblioteki | GPU NVIDIA, wersje PyTorch z 2021 r. | Apple M3 (MPS/CPU), aktualne wersje (komórka 0) | bardzo mały – inne implementacje operacji numerycznych |
| Preprocessing | `Resize(224, bicubic)` z PIL | ta sama funkcja `preprocess`, ale nowsza wersja PIL/torchvision | mały – obrazy 32×32 są powiększane 7×, więc szczegóły interpolacji mają znaczenie |
| Prompty | 18 szablonów z `data/prompts.md` | identyczne | brak (zweryfikowane w kodzie) |
| Zbiór testowy | oficjalny test CIFAR | oficjalny test przez `torchvision` (10 000 obrazów) | brak |
| Checkpoint | ViT-B/32 opisany w artykule | ViT-B/32 udostępniony publicznie | możliwa niewielka różnica, jeśli opublikowane wagi nie są identyczne z użytymi w Tabeli 11 |
| Losowość | — | brak (inferencja deterministyczna) | brak |

Interpretację uzyskanych liczb podsumowuje automatycznie poniższa komórka.

In [ ]:
print("PODSUMOWANIE REPRODUKCJI\n" + "=" * 60)
for name, r in RESULTS.items():
    diff = r["top1"] - PAPER_TOP1[name]
    in_ci = r["ci_low"] <= PAPER_TOP1[name] <= r["ci_high"]
    print(f"{name}: top-1 = {r['top1']:.2f}% (artykuł {PAPER_TOP1[name]}%), różnica {diff:+.2f} p.p., "
          f"95% CI [{r['ci_low']:.2f}, {r['ci_high']:.2f}]")
    if abs(diff) <= 1.0:
        verdict = "wynik odtworzony (różnica ≤ 1 p.p.)"
    elif abs(diff) <= 2.0:
        verdict = "wynik odtworzony z niewielką różnicą (1–2 p.p.) – patrz tabela przyczyn"
    else:
        verdict = "różnica > 2 p.p. – wymaga dodatkowej analizy (preprocessing, prompty, checkpoint)"
    print(f"   → {verdict}; wynik z artykułu {'mieści się' if in_ci else 'NIE mieści się'} w 95% CI\n")

print(f"Środowisko: Python {ENV['python']}, torch {ENV['torch']}, torchvision {ENV['torchvision']}, "
      f"urządzenie: {ENV['device']}")
print("Artefakty w katalogu results/:", sorted(p.name for p in RESULTS_DIR.iterdir()))

## 9. Podsumowanie i dalsze kroki

* Notatnik odtwarza procedurę zero-shot autorów: oficjalny checkpoint ViT-B/32, oficjalny `preprocess`,
  18 szablonów i nazwy klas z `data/prompts.md`, normalizacja L2, podobieństwo kosinusowe, top-1 accuracy na oficjalnych zbiorach testowych.
* Wszystkie wyniki użyte w prezentacji są zapisane w `results/` (CSV i PNG) i dają się odtworzyć przez *Restart & Run All*.
* **Konsultacja 3:** porównanie wariantów promptów – **A:** sama nazwa klasy, **B:** pojedynczy szablon, **C:** ensemble szablonów –
  z bootstrapem i przedziałami ufności. Embeddingi obrazów z `cache/` zostaną użyte ponownie, więc warianty promptów
  będą liczone szybko i na identycznych reprezentacjach obrazów (zmieniamy tylko prompt).

**Literatura**
* Radford A., Kim J.W., Hallacy C. i in. (2021). *Learning Transferable Visual Models From Natural Language Supervision*. ICML 2021. https://arxiv.org/abs/2103.00020
* Krizhevsky A. (2009). *Learning Multiple Layers of Features from Tiny Images*. Technical Report, University of Toronto.
* Oficjalne repozytorium CLIP (licencja MIT): https://github.com/openai/CLIP · szablony: https://github.com/openai/CLIP/blob/main/data/prompts.md